In [ ]:
from scipy.sparse import load_npz
from pathlib import Path

import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

import joblib
import plotly.express as px
import matplotlib.pyplot as plt

In [ ]:
processed_dir = Path("../data/processed")

tfidf_matrix = load_npz(processed_dir / "tfidf_matrix.npz")
ids = pd.read_csv(processed_dir / "tfidf_row_ids.csv")
df = pd.read_csv(processed_dir / "sample_clean.csv")

# Il faut s'assurer que la matrice TF-IDF et les identifiants des lignes correspondent
# bien aux produits du DataFrame original.
assert tfidf_matrix.shape[0] == len(ids)
assert ids["id"].is_unique
assert ids["id"].equals(df["id"])

print("Dimensions :", tfidf_matrix.shape)

In [ ]:
dbscan = DBSCAN(
    eps=0.5,
    min_samples=5,
    metric="cosine"
)

labels = dbscan.fit_predict(tfidf_matrix)

# Les résultats restent associés aux identifiants exportés.
clusters = ids.copy()
clusters["cluster"] = labels

display(clusters["cluster"].value_counts().sort_index())

clusters_nb = clusters.loc[
    clusters["cluster"] != -1, "cluster"
].nunique()

print("Nombre de clusters :", clusters_nb)

In [ ]:
without_noise = labels != -1

X_eval = tfidf_matrix[without_noise]
labels_evalues = labels[without_noise]

clusters_nb = len(np.unique(labels_evalues))

score = silhouette_score(
    X_eval,
    labels_evalues,
    metric="cosine"
)

print(f"Silhouette hors bruit : {score:.3f}")
print(f"Produits évalués : {without_noise.sum()} / {len(labels)}")
print("Noise_pct :", (labels == -1).mean() * 100)

In [ ]:
projection_2d = TruncatedSVD(n_components=2, random_state=42)

In [ ]:
projection_2d

In [ ]:
coordinates = projection_2d.fit_transform(tfidf_matrix)
visualisation = pd.DataFrame({
    "SVD 1": coordinates[:, 0],
    "SVD 2": coordinates[:, 1],
    "id": ids["id"].to_numpy(),
    "cluster": [
        "noise" if label == -1 else f"Cluster {label}"
        for label in labels
    ]
})

In [ ]:
cores = np.zeros(len(labels), dtype=bool)
cores[dbscan.core_sample_indices_] = True

visualisation["status"] = np.where(
    labels == -1,
    "noise",
    np.where(cores, "Core", "Border")
)

fig = px.scatter(
    visualisation,
    x="SVD 1",
    y="SVD 2",
    color="cluster",
    symbol="status",
    symbol_map={"Core": "circle", "Border": "diamond", "Noise": "x"},
    color_discrete_map={"Bruit": "lightgray"},
    hover_data=["id", "statut"],
    opacity=0.7,
    title="DBSCAN — clusters et vrais points cœur",
    height=600,
    width=600
)

fig.show()

In [ ]:
tsne = TSNE(
    n_components=2,
    metric="cosine",
    perplexity=30,
    init="random",
    learning_rate="auto",
    random_state=42
)

coord_tsne = tsne.fit_transform(tfidf_matrix.toarray())

visu_tsne = visualisation.copy()
visu_tsne["t-SNE 1"] = coord_tsne[:, 0]
visu_tsne["t-SNE 2"] = coord_tsne[:, 1]

fig = px.scatter(
    visu_tsne,
    x="t-SNE 1",
    y="t-SNE 2",
    color="cluster",
    symbol="statut",
    symbol_map={"Core": "circle", "Border": "diamond", "Noise": "x"},
    color_discrete_map={"Noise": "lightgray"},
    hover_data=["id", "status"],
    opacity=0.7,
    title="DBSCAN — visualisation t-SNE, distance cosinus",
    height=600,
    width=600
)

fig.show()

In [ ]:
tsne_3d = TSNE(
    n_components=3,
    metric="cosine",
    perplexity=30,
    init="random",
    learning_rate="auto",
    random_state=42
)

coord_3d = tsne_3d.fit_transform(tfidf_matrix.toarray())

visu_3d = visualisation.copy()
visu_3d[["t-SNE 1", "t-SNE 2", "t-SNE 3"]] = coord_3d

fig = px.scatter_3d(
    visu_3d,
    x="t-SNE 1",
    y="t-SNE 2",
    z="t-SNE 3",
    color="cluster",
    symbol="status",
    symbol_map={"Core": "circle", "Border": "diamond", "Noise": "x"},
    color_discrete_map={"Noise": "lightgray"},
    hover_data=["id", "status"],
    opacity=0.7,
    title="DBSCAN — projection t-SNE en 3D",
    height=600,
    width=600
)

fig.update_traces(marker_size=4)
fig.update_layout(scene_aspectmode="data")
fig.show()

In [ ]:
vectorizer = joblib.load("../models/tfidf_vectorizer.joblib")
mots = vectorizer.get_feature_names_out()

# Retrouver les descriptions dans l'ordre exact de la matrice
source = pd.read_csv(
    processed_dir / "sample_clean.csv"
).set_index("id")

reading = ids.copy()
reading["cluster"] = labels
reading["text"] = source.loc[
    ids["id"], "text"
].to_numpy()

assert tfidf_matrix.shape == (len(reading), len(words))

In [ ]:
cluster_ids = np.unique(labels[labels != -1]) # Parcours tous les clusters sauf le bruit (-1)

for cluster_id in cluster_ids:
    mask = labels == cluster_id
    assert cluster_id != -1 and mask.any()

    X_cluster = tfidf_matrix[mask]

    # Poids TF-IDF moyen de chaque mot dans ce groupe
    weight = np.asarray(X_cluster.mean(axis=0)).ravel()
    top = np.argsort(weight)[::-1][:12]
    top = top[weight[top] > 0]

    words_cluster = pd.DataFrame({
        "word": words[top],
        "Mean weight TF-IDF": weight[top]
    })

    fig = px.bar(
        words_cluster,
        x="Mean_weight TF-IDF",
        y="word",
        orientation="h",
        title=f"Cluster {cluster_id} — mots dominants",
        height=400,
        width=600
    )
    fig.update_yaxes(categoryorder="total ascending")
    fig.show()

    # Descriptions les plus proches du centre moyen en distance cosinus
    center = np.asarray(X_cluster.mean(axis=0))
    similarites = cosine_similarity(X_cluster, center).ravel()

    positions = np.flatnonzero(mask)
    selection = positions[np.argsort(similarites)[::-1][:3]]

    print(f"Cluster {cluster_id} : {len(positions)} produits")

    with pd.option_context("display.max_colwidth", None):
        display(reading.iloc[selection][["id", "text"]])